In [1]:
#!/usr/bin/env python3
"""
Step 1: Dual Vocabulary Builder Script

This script builds both full and filtered vocabularies simultaneously,
allowing comparison of SPLADE performance with and without Spanish stopwords.
Perfect for A/B testing different vocabulary approaches.

"""

import sys
from pathlib import Path
from typing import List
import re

# Add project root to path
project_root = Path('/work')
sys.path.append(str(project_root))

# Import utilities
from src.utils.data_utils import load_documents

# Import enhanced vocabulary components
from sandbox.splade.core.vocabulary import DualVocabularyBuilder

def load_corpus_documents(file_base: str) -> tuple:
    """
    Load both texto and resumen documents.
    
    Args:
        file_base: Base filename for documents
        
    Returns:
        Tuple of (texto_docs, resumen_docs)
    """
    print("Loading document collections...")
    
    # Load texto documents
    texto_path = Path(f'/work/data/processed/{file_base}_texto.pkl')
    texto_docs = load_documents(texto_path)
    
    if texto_docs is None:
        raise FileNotFoundError(f"Could not load texto documents from {texto_path}")
    
    # Load resumen documents
    resumen_path = Path(f'/work/data/processed/{file_base}_resumen.pkl')
    resumen_docs = load_documents(resumen_path)
    
    if resumen_docs is None:
        raise FileNotFoundError(f"Could not load resumen documents from {resumen_path}")
    
    print(f"Loaded {len(texto_docs)} texto documents")
    print(f"Loaded {len(resumen_docs)} resumen documents")
    
    return texto_docs, resumen_docs

def extract_compound_terms_from_documents(documents):
    """
    Extract all compound terms (multi-word terms) from a list of Document objects.
    
    Args:
        documents: List of Document objects with metadata containing 'parameters'
        
    Returns:
        List of unique compound terms, sorted alphabetically
    """
    compound_terms = set()  # Use set to avoid duplicates
    
    def is_compound_term(text):
        """Check if a term has more than one word"""
        if not text or not isinstance(text, str):
            return False
        
        # Clean the text: remove extra spaces and trim
        clean_text = re.sub(r'\s+', ' ', text.strip())
        
        # Check if it contains more than one word
        words = [word for word in clean_text.split(' ') if word]
        return len(words) > 1
    
    def add_if_compound(text):
        """Add text to compound_terms set if it's a compound term"""
        if is_compound_term(text):
            clean_text = re.sub(r'\s+', ' ', text.strip())
            compound_terms.add(clean_text)
    
    # Iterate through each Document object
    for document in documents:
        # Get parameters from document metadata
        parameters = document.metadata.get('parameters', {})
        
        # Skip if no parameters or parameters is empty
        if not parameters:
            continue
        
        # Iterate through each parameter (A, B, C, etc.)
        for param_key, parameter in parameters.items():
            
            # Check parameter label
            if 'label' in parameter:
                add_if_compound(parameter['label'])
            
            # Check values array
            if 'values' in parameter and isinstance(parameter['values'], list):
                for value_item in parameter['values']:
                    # Check value field
                    if 'value' in value_item:
                        add_if_compound(value_item['value'])
    
    # Convert set to sorted list
    return sorted(list(compound_terms))

def analyze_dual_corpus_characteristics(texto_docs, resumen_docs, dual_builder):
    """
    Analyze corpus characteristics with both tokenizers.
    
    Args:
        texto_docs: Texto document collection
        resumen_docs: Resumen document collection  
        dual_builder: DualVocabularyBuilder instance
    """
    print("\n" + "="*60)
    print("DUAL CORPUS ANALYSIS")
    print("="*60)
    
    # Analyze both collections with both tokenizers
    collections = [
        ("Texto", texto_docs),
        ("Resumen", resumen_docs)
    ]
    
    tokenizers = [
        ("Full (with stopwords)", dual_builder.tokenizer_full),
        ("Filtered (no stopwords)", dual_builder.tokenizer_filtered)
    ]
    
    for coll_name, docs in collections:
        print(f"\n{coll_name} Collection Analysis:")
        
        # Sample a few documents for analysis
        sample_size = min(1000, len(docs))
        sample_docs = docs[:sample_size]
        
        for tok_name, tokenizer in tokenizers:
            total_tokens = 0
            total_chars = 0
            word_lengths = []
            
            for doc in sample_docs:
                text = doc.text if hasattr(doc, 'text') else str(doc)
                tokens = tokenizer.tokenize(text)
                
                total_tokens += len(tokens)
                total_chars += len(text)
                word_lengths.extend([len(token) for token in tokens])
            
            avg_tokens = total_tokens / len(sample_docs)
            avg_chars = total_chars / len(sample_docs)
            avg_word_len = sum(word_lengths) / len(word_lengths) if word_lengths else 0
            
            print(f"  {tok_name}:")
            print(f"    Documents analyzed: {len(sample_docs):,}")
            print(f"    Avg tokens per doc: {avg_tokens:.1f}")
            print(f"    Avg chars per doc: {avg_chars:.1f}")
            print(f"    Avg word length: {avg_word_len:.1f}")
            
            # Show sample tokens
            if sample_docs:
                sample_text = sample_docs[0].text if hasattr(sample_docs[0], 'text') else str(sample_docs[0])
                sample_tokens = tokenizer.tokenize(sample_text)[:15]
                print(f"    Sample tokens: {', '.join(sample_tokens)}")


def build_dual_vocabulary_jupyter(
    file_base: str,
    vocab_size: int = 50000,
    min_freq: int = 1,
    output_dir: str = "/work/sandbox/splade/vocabularies",
    analyze_only: bool = False,
):
    """
    Build both full and filtered vocabularies for SPLADE - Jupyter-friendly version.
    
    Args:
        file_base: Base filename for document files
        vocab_size: Maximum vocabulary size
        min_freq: Minimum word frequency to include
        output_dir: Output directory for vocabulary files
        analyze_only: Only analyze corpus, don't build vocabularies
        custom_compound_terms: Additional compound terms to preserve (e.g., ['bajo vías', 'volumen escaso'])
    """
    
    print("="*60)
    print("DUAL WORD-LEVEL VOCABULARY BUILDER")
    print("="*60)
    print(f"File base: {file_base}")
    print(f"Vocabulary size: {vocab_size:,}")
    print(f"Min frequency: {min_freq}")
    print(f"Building: Full + Filtered vocabularies")
    
    try:
        # Load documents
        texto_docs, resumen_docs = load_corpus_documents(file_base)
        
        # Combine all documents for vocabulary building
        all_docs = texto_docs + resumen_docs
        print(f"Total documents for vocabularies: {len(all_docs):,}")

        # Extract compound terms
        custom_compound_terms = extract_compound_terms_from_documents(texto_docs)
        print(f"Compound terms: {custom_compound_terms}")
        
        # Initialize dual vocabulary builder
        dual_builder = DualVocabularyBuilder(
            min_freq=min_freq,
            max_vocab_size=vocab_size,
            include_special_tokens=True,
            compound_terms=custom_compound_terms
        )
        
        # Add custom compound terms if provided
        if custom_compound_terms:
            dual_builder.tokenizer_full.add_compound_terms(custom_compound_terms)
            dual_builder.tokenizer_filtered.add_compound_terms(custom_compound_terms)
            print(f"Added {len(custom_compound_terms)} custom compound terms")
            print(f"Compound terms: {custom_compound_terms}")
            
            # Test compound term tokenization
            print(f"\n🔧 Testing compound term tokenization:")
            test_text = " ".join(custom_compound_terms[:3])  # Test first 3 terms
            full_tokens = dual_builder.tokenizer_full.tokenize(test_text)
            filtered_tokens = dual_builder.tokenizer_filtered.tokenize(test_text)
            print(f"  Test text: '{test_text}'")
            print(f"  Full tokens: {full_tokens}")
            print(f"  Filtered tokens: {filtered_tokens}")
        else:
            print("Using default compound terms from construction domain")
        
        # Analyze corpus characteristics with both tokenizers
        analyze_dual_corpus_characteristics(texto_docs, resumen_docs, dual_builder)
        
        # Analyze corpus for statistics
        print(f"\nAnalyzing full corpus with both vocabularies...")
        full_stats, filtered_stats = dual_builder.analyze_corpus(all_docs)
        
        if analyze_only:
            print("\nCorpus analysis complete (analyze_only mode)")
            dual_builder.print_comparison_stats(full_stats, filtered_stats)
            return dual_builder, full_stats, filtered_stats
        
        # Build both vocabularies
        print(f"\nBuilding both vocabularies...")
        dual_builder.build_vocabularies(all_docs)
        
        # Calculate coverage on different document sets
        print(f"\nCalculating vocabulary coverage for both approaches...")
        
        # Test on samples for speed
        sample_texto = texto_docs[:1000] if len(texto_docs) > 1000 else texto_docs
        sample_resumen = resumen_docs[:1000] if len(resumen_docs) > 1000 else resumen_docs
        
        texto_coverage_full, texto_coverage_filtered = dual_builder.calculate_coverage(sample_texto)
        resumen_coverage_full, resumen_coverage_filtered = dual_builder.calculate_coverage(sample_resumen)
        
        # Print detailed comparison statistics
        dual_builder.print_comparison_stats(full_stats, filtered_stats)
        
        # Print coverage comparison
        print(f"\n" + "="*60)
        print("COVERAGE COMPARISON")
        print("="*60)
        print(f"{'Dataset':<15} {'Full Vocab':<15} {'Filtered Vocab':<15} {'Difference':<15}")
        print("-" * 60)
        
        texto_diff = texto_coverage_full['coverage_ratio'] - texto_coverage_filtered['coverage_ratio']
        resumen_diff = resumen_coverage_full['coverage_ratio'] - resumen_coverage_filtered['coverage_ratio']
        
        print(f"{'Texto':<15} {texto_coverage_full['coverage_ratio']:.3f}{'':<9} {texto_coverage_filtered['coverage_ratio']:.3f}{'':<9} {texto_diff:+.3f}")
        print(f"{'Resumen':<15} {resumen_coverage_full['coverage_ratio']:.3f}{'':<9} {resumen_coverage_filtered['coverage_ratio']:.3f}{'':<9} {resumen_diff:+.3f}")
        
        # Save both vocabularies
        output_path = Path(output_dir)
        dual_builder.save_vocabularies(output_path, file_base)
        
        print(f"\n✅ Dual vocabulary building complete!")
        print(f"📁 Vocabularies saved to: {output_path}")
        
        # Summary table
        print(f"\n📊 VOCABULARY SUMMARY:")
        print(f"{'Approach':<20} {'Size':<10} {'Coverage (Texto)':<18} {'Coverage (Resumen)':<20}")
        print("-" * 68)
        print(f"{'Full (w/ stopwords)':<20} {len(dual_builder.vocab_full.word_to_id):<10} {texto_coverage_full['coverage_ratio']:.1%}{'':<10} {resumen_coverage_full['coverage_ratio']:.1%}")
        print(f"{'Filtered (no stops)':<20} {len(dual_builder.vocab_filtered.word_to_id):<10} {texto_coverage_filtered['coverage_ratio']:.1%}{'':<10} {resumen_coverage_filtered['coverage_ratio']:.1%}")
        
        # Show example encodings for both vocabularies
        print(f"\n🔍 Example encodings comparison:")
        sample_texts = [
            "canalización de tubos",
            "ejecución y hormigonada", 
            "galvanizado mm",
            "los tubos de polietileno"
        ]
        
        print(f"{'Text':<25} {'Full Vocab':<15} {'Filtered Vocab':<15}")
        print("-" * 55)
        
        for text in sample_texts:
            full_tokens = dual_builder.vocab_full.tokenizer.tokenize(text)
            filtered_tokens = dual_builder.vocab_filtered.tokenizer.tokenize(text)
            full_ids = dual_builder.vocab_full.encode(text)
            filtered_ids = dual_builder.vocab_filtered.encode(text)
            
            print(f"'{text}'")
            print(f"  Tokens: {full_tokens} -> {filtered_tokens}")
            print(f"  IDs:    {full_ids} -> {filtered_ids}")
            print()
            
        # Recommendations
        print(f"🎯 RECOMMENDATIONS:")
        
        vocab_reduction = len(dual_builder.vocab_full.word_to_id) - len(dual_builder.vocab_filtered.word_to_id)
        if vocab_reduction > 10:
            print(f"✓ Significant vocabulary reduction: -{vocab_reduction} words")
            print(f"✓ Consider filtered vocabulary for better focus on technical terms")
        
        if texto_coverage_filtered['coverage_ratio'] >= 0.95 and resumen_coverage_filtered['coverage_ratio'] >= 0.95:
            print(f"✓ Excellent coverage maintained in filtered vocabulary")
            print(f"✓ Minimal information loss from stopword removal")
        
        if vocab_reduction < 5:
            print(f"! Small vocabulary reduction - stopwords may not be frequent in your domain")
            print(f"! Consider using full vocabulary")
        
        print(f"\n💡 Next steps:")
        print(f"  1. Train SPLADE models with both vocabularies")
        print(f"  2. Compare retrieval performance on your test set")
        print(f"  3. Choose the vocabulary that gives better results")
            
        return dual_builder, full_stats, filtered_stats
    
    except Exception as e:
        print(f"❌ Error: {e}")
        raise


# Convenience functions for quick testing
def quick_analysis(file_base: str = "OEB", custom_compounds: List[str] = None):
    """Quick corpus analysis without building vocabularies."""
    return build_dual_vocabulary_jupyter(
        file_base=file_base,
        analyze_only=True,
        custom_compound_terms=custom_compounds
    )

def build_vocabularies(file_base: str = "OEB", vocab_size: int = 50000, min_freq: int = 1, custom_compounds: List[str] = None):
    """Build both vocabularies with standard settings."""
    return build_dual_vocabulary_jupyter(
        file_base=file_base,
        vocab_size=vocab_size,
        min_freq=min_freq,
        analyze_only=False,
        custom_compound_terms=custom_compounds
    )


In [2]:
vocab_builder, full_stats, filtered_stats = build_dual_vocabulary_jupyter(
        file_base="OEB",  # Replace with your actual file base
        vocab_size=50000,
        min_freq=1,
        output_dir="/work/sandbox/splade/vocabularies",
        analyze_only=False,  # Set to True if you only want to analyze,  
    )

DUAL WORD-LEVEL VOCABULARY BUILDER
File base: OEB
Vocabulary size: 50,000
Min frequency: 1
Building: Full + Filtered vocabularies
Loading document collections...
Loaded 47514 texto documents
Loaded 47514 resumen documents
Total documents for vocabularies: 95,028
Compound terms: ['0,80 a 1,00 m', '1 o 2', '1,10 m', "3 1/2''", '3 <= i < 5 horas', 'BANDA DE MANTENIMIENTO', 'Bajo vías', 'CONDICIONES DE EJECUCIÓN', 'Con topo', 'Cruce de carretera', 'Cualquier condición de ejecución', 'Cualquier franja horaria', 'Cualquier franja horaria excepcional', 'Diurno Excepcional', 'Diurno excepcional', 'No aplica', 'No necesita intervalo', 'Nocturno Excepcional', 'Nocturno excepcional', 'Nº TUBOS', 'Suministro y montaje', 'TIPO DE ACCIÓN', 'TIPO DE TERRENO', 'Volumen escaso', 'Volumen relevante', 'con reposición', 'de cantera', 'de la traza', 'hasta 0,80 m', 'i < 3 horas', 'i >= 5 horas', 'i< 3 horas', 'sin reposición']
Added 33 compound terms to tokenizer
Added 33 compound terms to tokenizer
Added 